In [3]:
# -*- coding: utf-8 -*-
r"""
Fetch all pom.xml files for a list of GitHub repos and save them with your naming convention:
  owner.repo__maven++pom.xml
  owner.repo__maven++pom__2.xml
  ...

Inputs:
- URL list CSV at:  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\URL_List.csv
  (column may be 'html_url' or 'url' or 'repo_url' or 'full_name' OR any column containing GitHub URLs or owner/repo strings)

Tokens:
- Reads tokens from (first found):
    C:\GitHub\Android-Mobile-Apps\all_tokens.env
    C:\GitHub\Android-Mobile-Apps\All_Tokens.env
  Keys like: GITHUB_TOKEN_1=ghp_xxx, GITHUB_TOKEN_2=...

Output:
- All POMs saved to:
    C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_POMs
"""

import os, re, io, json, time, random, base64
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd

# ======================= CONFIG =======================
BASE_DIR     = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10")
URL_LIST_CSV = BASE_DIR / "URL_List.csv"
OUT_DIR      = BASE_DIR / "All_POMs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TOKENS_ENV_PATHS = [
    Path(r"C:\GitHub\Android-Mobile-Apps\all_tokens.env"),
    Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env"),
]

VERBOSE = True
REQUEST_TIMEOUT      = 30
REQUESTS_MAX_RETRIES = 3
BACKOFF_BASE_SEC     = 2.0

# ======================= LOG =======================
def ts():
    import datetime as _dt
    return _dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def info(msg): print(f"[{ts()}] [INFO] {msg}", flush=True)
def warn(msg): print(f"[{ts()}] [WARN] {msg}", flush=True)
def note(msg): print(f"[{ts()}] {msg}", flush=True)

# ======================= TOKENS =======================
def load_tokens_from_env_file(path: Path) -> List[str]:
    tokens = {}
    if path and path.exists():
        text = path.read_text(encoding="utf-8", errors="ignore")
        for raw in text.splitlines():
            line = raw.strip()
            if not line or line.startswith(("#",";")) or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k.upper().startswith("GITHUB_TOKEN_"):
                try:
                    idx = int(k.split("_")[-1])
                except Exception:
                    idx = 999
                tokens[idx] = v
    return [tokens[i] for i in sorted(tokens.keys()) if tokens.get(i)]

def load_all_tokens(paths: List[Path]) -> List[str]:
    for p in paths:
        toks = load_tokens_from_env_file(p)
        if toks:
            info(f"Loaded {len(toks)} GitHub token(s) from {p}")
            return toks
        else:
            warn(f"No tokens found at {p}")
    warn("Proceeding without tokens (very rate-limited).")
    return []

class TokenRotator:
    def __init__(self, tokens: List[str]):
        self.tokens = tokens or []
        self.i = 0
    def current(self) -> Optional[str]:
        return self.tokens[self.i] if self.tokens else None
    def rotate(self):
        if self.tokens:
            self.i = (self.i + 1) % len(self.tokens)
    def headers(self, allow_auth=True):
        base = {"Accept": "application/vnd.github+json",
                "User-Agent": "pom-fetcher/1.0"}
        tok = self.current()
        if allow_auth and tok:
            base["Authorization"] = f"Bearer {tok}"
        return base

# ======================= HTTP =======================
def gh_get(url: str, rot: TokenRotator, allow_auth=True, timeout=REQUEST_TIMEOUT):
    import requests
    tries = 0
    while True:
        tries += 1
        resp = requests.get(url, headers=rot.headers(allow_auth=allow_auth), timeout=timeout)
        diag = {
            "status": resp.status_code,
            "rate_remaining": resp.headers.get("X-RateLimit-Remaining"),
            "rate_reset": resp.headers.get("X-RateLimit-Reset"),
            "url": url,
        }
        resp._diag = diag

        if resp.status_code in (200, 201, 204):
            return resp

        if resp.status_code in (403, 429):
            warn(f"Rate/403 on {url}. Rotating token. Diag={diag}")
            rot.rotate()
            if tries >= REQUESTS_MAX_RETRIES * max(1, len(rot.tokens)):
                return resp
            sleep_s = BACKOFF_BASE_SEC * (2 ** (tries - 1)) + random.random()
            time.sleep(min(sleep_s, 30))
            continue

        if resp.status_code == 401 and allow_auth:
            return gh_get(url, rot, allow_auth=False, timeout=timeout)

        return resp

# ======================= HELPERS =======================
GH_URL_RE = re.compile(r'https?://(?:www\.)?github\.com/([^/\s]+)/([^/\s#?]+)', re.I)
FULLNAME_RE = re.compile(r'^[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+$')

def owner_repo_from_url(u: str) -> Tuple[Optional[str], Optional[str]]:
    m = GH_URL_RE.search(u or "")
    if m:
        return m.group(1).lower(), m.group(2).lower()
    if FULLNAME_RE.match((u or "").strip()):
        o, r = u.strip().split("/", 1)
        return o.lower(), r.lower()
    return None, None

def sanitize_full_name(owner: str, repo: str) -> str:
    o = (owner or "").replace(" ", "").lower()
    r = (repo or "").replace(" ", "").lower()
    return f"{o}.{r}"

def read_url_list(csv_path: Path) -> List[Tuple[str,str]]:
    if not csv_path.exists():
        raise FileNotFoundError(f"URL list not found: {csv_path}")

    # Try CSV via pandas
    try:
        df = pd.read_csv(csv_path)
        cols = {c.lower(): c for c in df.columns}
        note(f"Detected columns: {list(df.columns)}")

        # 1) Preferred explicit columns
        if "full_name" in cols:
            seq = df[cols["full_name"]].dropna().astype(str).tolist()
            pairs = []
            for val in seq:
                o, r = owner_repo_from_url(val)
                if o and r:
                    pairs.append((o, r))
            if pairs:
                return _dedup_preserve(pairs)

        for key in ["html_url", "url", "repo_url", "hurl", "repository_url", "clone_url", "ssh_url", "git_url"]:
            if key in cols:
                seq = df[cols[key]].dropna().astype(str).tolist()
                pairs = []
                for url in seq:
                    o, r = owner_repo_from_url(url)
                    if o and r:
                        pairs.append((o, r))
                if pairs:
                    return _dedup_preserve(pairs)

        # 2) Fallback: scan EVERY cell for a GitHub URL or owner/repo
        pairs = []
        for _, row in df.iterrows():
            found = None
            for c in df.columns:
                val = str(row[c])
                o, r = owner_repo_from_url(val)
                if o and r:
                    found = (o, r); break
            if found:
                pairs.append(found)
        if pairs:
            return _dedup_preserve(pairs)

        # 3) Last resort: use the first column as a list
        if not df.empty:
            first_col_vals = df.iloc[:, 0].dropna().astype(str).tolist()
            pairs = []
            for v in first_col_vals:
                o, r = owner_repo_from_url(v)
                if o and r:
                    pairs.append((o, r))
            if pairs:
                return _dedup_preserve(pairs)

    except Exception as e:
        warn(f"pandas could not parse CSV ({e}); trying line-by-line text fallback")

    # Plain text fallback (one per line)
    lines = csv_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    pairs = []
    for ln in lines:
        o, r = owner_repo_from_url(ln.strip())
        if o and r:
            pairs.append((o, r))
    if pairs:
        return _dedup_preserve(pairs)

    raise ValueError(
        "Could not find any GitHub owner/repo entries in the list. "
        "Please include a column with GitHub URLs or values like 'owner/repo'."
    )

def _dedup_preserve(pairs: List[Tuple[str,str]]) -> List[Tuple[str,str]]:
    seen, out = set(), []
    for o, r in pairs:
        key = f"{o}/{r}"
        if key not in seen:
            seen.add(key); out.append((o, r))
    return out

# ======================= GITHUB OPS =======================
def get_default_branch(rot: TokenRotator, owner: str, repo: str) -> Optional[str]:
    resp = gh_get(f"https://api.github.com/repos/{owner}/{repo}", rot, allow_auth=True)
    if resp.status_code != 200:
        warn(f"Repo meta failed {owner}/{repo}: {resp._diag}")
        return None
    try:
        return resp.json().get("default_branch") or "main"
    except Exception:
        return "main"

def get_branch_head_sha(rot: TokenRotator, owner: str, repo: str, branch: str) -> Optional[str]:
    resp = gh_get(f"https://api.github.com/repos/{owner}/{repo}/git/refs/heads/{branch}", rot, allow_auth=True)
    if resp.status_code == 200:
        try:
            return resp.json()["object"]["sha"]
        except Exception:
            pass
    resp = gh_get(f"https://api.github.com/repos/{owner}/{repo}/commits/{branch}", rot, allow_auth=True)
    if resp.status_code == 200:
        try:
            return resp.json()["sha"]
        except Exception:
            pass
    warn(f"Could not resolve branch SHA for {owner}/{repo}@{branch}: {resp._diag}")
    return None

def list_tree_paths(rot: TokenRotator, owner: str, repo: str, tree_sha: str) -> List[str]:
    resp = gh_get(f"https://api.github.com/repos/{owner}/{repo}/git/trees/{tree_sha}?recursive=1", rot, allow_auth=True)
    if resp.status_code != 200:
        warn(f"Tree listing failed for {owner}/{repo}@{tree_sha}: {resp._diag}")
        return []
    data = resp.json()
    if data.get("truncated"):
        warn(f"Tree listing truncated for {owner}/{repo} — results may be partial.")
    paths = []
    for item in data.get("tree", []):
        if item.get("type") == "blob":
            p = (item.get("path") or "").lower()
            if p.endswith("/pom.xml") or p == "pom.xml":
                paths.append(item.get("path"))
    return paths

def fetch_pom_bytes(rot: TokenRotator, owner: str, repo: str, path: str, ref: str) -> Optional[bytes]:
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}?ref={ref}"
    resp = gh_get(url, rot, allow_auth=True)
    if resp.status_code != 200:
        warn(f"Content fetch failed: {owner}/{repo}:{path}@{ref}: {resp._diag}")
        return None
    try:
        j = resp.json()
        if isinstance(j, dict) and j.get("encoding") == "base64" and "content" in j:
            return base64.b64decode(j["content"])
        if isinstance(j, dict) and "content" in j and not j.get("encoding"):
            return (j["content"] or "").encode("utf-8", errors="ignore")
    except Exception as e:
        warn(f"Content decode error for {owner}/{repo}:{path} — {e}")
    return None

def next_pom_filename(idx: int) -> str:
    return "pom.xml" if idx == 1 else f"pom__{idx}.xml"

def save_poms_for_repo(rot: TokenRotator, owner: str, repo: str) -> int:
    branch = get_default_branch(rot, owner, repo)
    if not branch:
        return 0
    sha = get_branch_head_sha(rot, owner, repo, branch)
    if not sha:
        return 0
    paths = list_tree_paths(rot, owner, repo, sha)
    if not paths:
        if VERBOSE: note(f"{owner}/{repo}: no pom.xml found")
        return 0

    saved = 0
    stem = sanitize_full_name(owner, repo)  # owner.repo
    for i, path in enumerate(paths, start=1):
        data = fetch_pom_bytes(rot, owner, repo, path, branch)
        if not data:
            continue
        out_name = f"{stem}__maven++{next_pom_filename(i)}"
        out_path = OUT_DIR / out_name
        try:
            out_path.write_bytes(data)
            saved += 1
            if VERBOSE:
                note(f"Saved {owner}/{repo}:{path} -> {out_path.name}")
        except Exception as e:
            warn(f"Write failed for {out_path}: {e}")
    return saved

# ======================= MAIN =======================
def main():
    tokens = load_all_tokens(TOKENS_ENV_PATHS)
    rot = TokenRotator(tokens)

    owner_repos = read_url_list(URL_LIST_CSV)
    info(f"Repos in URL list: {len(owner_repos)}")

    total_poms = 0
    for idx, (owner, repo) in enumerate(owner_repos, start=1):
        t0 = time.time()
        try:
            cnt = save_poms_for_repo(rot, owner, repo)
            total_poms += cnt
            dur = time.time() - t0
            note(f"[{idx}/{len(owner_repos)}] {owner}/{repo} -> {cnt} pom(s) in {dur:.1f}s")
        except Exception as e:
            warn(f"Unhandled error on {owner}/{repo}: {e}")

    info(f"DONE. Total pom.xml files saved: {total_poms} to {OUT_DIR}")

if __name__ == "__main__":
    main()


[2025-08-24 02:28:33] [INFO] Loaded 6 GitHub token(s) from C:\GitHub\Android-Mobile-Apps\all_tokens.env
[2025-08-24 02:28:33] Detected columns: ['github_url']
[2025-08-24 02:28:33] [INFO] Repos in URL list: 4697
[2025-08-24 02:28:34] jamplus/jamplus: no pom.xml found
[2025-08-24 02:28:34] [1/4697] jamplus/jamplus -> 0 pom(s) in 0.8s
[2025-08-24 02:28:35] samuelclay/newsblur: no pom.xml found
[2025-08-24 02:28:35] [2/4697] samuelclay/newsblur -> 0 pom(s) in 0.9s
[2025-08-24 02:28:36] connectbot/connectbot: no pom.xml found
[2025-08-24 02:28:36] [3/4697] connectbot/connectbot -> 0 pom(s) in 0.7s
[2025-08-24 02:28:36] pocmo/yaaic: no pom.xml found
[2025-08-24 02:28:36] [4/4697] pocmo/yaaic -> 0 pom(s) in 0.6s
[2025-08-24 02:28:37] ramblurr/anki-android: no pom.xml found
[2025-08-24 02:28:37] [5/4697] ramblurr/anki-android -> 0 pom(s) in 0.7s
[2025-08-24 02:28:38] xcsoar/xcsoar: no pom.xml found
[2025-08-24 02:28:38] [6/4697] xcsoar/xcsoar -> 0 pom(s) in 0.8s
[2025-08-24 02:28:40] Saved gr